# **Setup & Download Model Dlib**
Import semua library (OpenCV, dlib, skimage, sklearn, dll). Download model shape_predictor_68_face_landmarks.dat dari Dlib kalau belum ada — model ini dipakai nanti untuk mendeteksi 68 titik landmark wajah.

In [ ]:
import os
import time
import urllib.request
import bz2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import cv2
import dlib
from PIL import Image
from skimage.feature import hog, local_binary_pattern

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, auc)

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

model_path = "shape_predictor_68_face_landmarks.dat"
if not os.path.exists(model_path):
    print("Mengunduh model Dlib 68 Facial Landmarks...")
    url = "http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2"
    urllib.request.urlretrieve(url, model_path + ".bz2")
    with bz2.BZ2File(model_path + ".bz2", 'rb') as source, open(model_path, 'wb') as dest:
        dest.write(source.read())
    os.remove(model_path + ".bz2")
    print("Download model selesai!\n")
else:
    print("Model Dlib 68 Facial Landmarks sudah tersedia.\n")

Mengunduh model Dlib 68 Facial Landmarks...


# **Mount Google Drive & Path Dataset**
Menghubungkan Colab ke Google Drive, lalu menentukan path folder real dan ai di dalam dataset_final.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Tubes_RF/dataset_final'

real_path = os.path.join(base_path, 'real')
ai_path = os.path.join(base_path, 'ai')

print(f"Path Real Faces: {real_path}")
print(f"Path AI Faces: {ai_path}")

ValueError: mount failed

# **Load Gambar**
Fungsi load_images_only membaca semua gambar dari folder (grayscale, resize ke 128x128 kalau perlu), lalu memuat data Real (label 0) dan AI (label 1), digabung jadi array X dan y.

In [ ]:
IMG_SIZE = (128, 128)

def load_images_only(folder_path, label):
    images = []
    labels = []
    skipped_count = 0

    valid_extensions = ('.jpg', '.jpeg', '.png')
    file_names = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]

    for filename in tqdm(file_names, desc=f"Loading Label {label}"):
        img_path = os.path.join(folder_path, filename)
        img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img_gray is not None:
            # Jaga-jaga kalau ukurannya belum persis 128x128
            if img_gray.shape[:2] != IMG_SIZE:
                img_gray = cv2.resize(img_gray, IMG_SIZE)

            images.append(img_gray)
            labels.append(label)
        else:
            skipped_count += 1

    print(f"\nLabel {label} - Gagal dibaca: {skipped_count} gambar.")
    return images, labels

# Pemuatan Data (langsung load, tanpa deteksi wajah lagi)
print("Memuat dataset Wajah Asli (Real)...")
real_images, real_labels = load_images_only(real_path, label=0)

print("\nMemuat dataset Wajah Buatan (AI)...")
ai_images, ai_labels = load_images_only(ai_path, label=1)

# Menggabungkan Data
X = np.array(real_images + ai_images)
y = np.array(real_labels + ai_labels)
print(f"\nTotal data berhasil dimuat: {len(X)} gambar")

# **Split Train/Val/Test**
Membagi data jadi 70% Training, 20% Validasi, 10% Uji (dua tahap train_test_split, dengan stratify supaya proporsi Real:AI tetap seimbang di tiap split).

In [ ]:
# Memisahkan 70% Training dan 30% Sisanya (Validation + Testing)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Membagi 30% sisa tersebut menjadi 20% Validation dan 10% Testing
# Proporsi 2/3 dari 30% adalah 20%, dan 1/3 dari 30% adalah 10%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=(1/3), random_state=42, stratify=y_temp
)

print("--- Hasil Tahap 4: Pembagian Dataset ---")
print(f"Data Latih (Training 70%): {len(X_train)} gambar")
print(f"Data Validasi (Validation 20%): {len(X_val)} gambar")
print(f"Data Uji (Testing 10%): {len(X_test)} gambar")

# **Augmentasi Data Latih**
90% dari X_train diaugmentasi secara acak (flip horizontal, rotasi 15°, atau perubahan kecerahan), lalu digabungkan kembali ke X_train. Hanya data latih yang diaugmentasi — val/test tidak disentuh.

In [ ]:
#AUGMENTASI DATA LATIH & VISUALISASI OUTPUT
import cv2
import random
import matplotlib.pyplot as plt

# Menentukan 90% dari total data latih yang akan diaugmentasi
num_to_augment = int(0.9 * len(X_train))
np.random.seed(42)
indices_to_augment = np.random.choice(len(X_train), num_to_augment, replace=False)

augmented_images = []
augmented_labels = []

# Variabel bantuan untuk menyimpan sampel foto visualisasi
sample_aug_real = None
sample_aug_ai = None

for idx in indices_to_augment:
    img = X_train[idx]
    label = y_train[idx]

    #Memilih jenis augmentasi secara acak (1 = Flip, 2 = Rotasi, 3 = Kecerahan)
    aug_type = random.randint(1, 3)
    if aug_type == 1:
        aug_img = cv2.flip(img, 1)
    elif aug_type == 2:
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), 15, 1.0)
        aug_img = cv2.warpAffine(img, M, (w, h))
    else:
        aug_img = cv2.convertScaleAbs(img, alpha=1.0, beta=30)

    augmented_images.append(aug_img)
    augmented_labels.append(label)

    #Menyimpan 1 sampel Real dan 1 sampel AI untuk divisualisasikan
    if label == 0 and sample_aug_real is None:
        sample_aug_real = aug_img
    elif label == 1 and sample_aug_ai is None:
        sample_aug_ai = aug_img

#Menggabungkan gambar hasil augmentasi kembali ke dataset latih utama
X_train = np.vstack((X_train, np.array(augmented_images)))
y_train = np.hstack((y_train, np.array(augmented_labels)))

print(" Hasil Tahap 5: Augmentasi")
print(f"Jumlah sampel yang diaugmentasi (90% dari data latih): {num_to_augment} gambar")
print(f"Total data latih (X_train) sekarang menjadi: {len(X_train)} gambar\n")

#MENAMPILKAN OUTPUT FOTO (REAL & AI)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#Foto Wajah Asli (Real) - Hasil Augmentasi
axes[0].imshow(sample_aug_real, cmap='gray')
axes[0].set_title('Real - Hasil Augmentasi')
axes[0].axis('off')

#Foto Wajah AI - Hasil Augmentasi
axes[1].imshow(sample_aug_ai, cmap='gray')
axes[1].set_title('AI - Hasil Augmentasi')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# **Cek Distribusi Setelah Augmentasi**

In [ ]:
print("=DISTRIBUSI DATASET SETELAH AUGMENTASI")
print(f"Total Data Latih (Training)  : {len(X_train)} gambar")
print(f"Total Data Validasi (Validation) : {len(X_val)} gambar")
print(f"Total Data Uji (Testing)     : {len(X_test)} gambar")
print("-" * 45)
print(f"Total Keseluruhan Data       : {len(X_train) + len(X_val) + len(X_test)} gambar")

# **Ekstraksi Fitur LBP (Local Binary Pattern)**
Menghitung tekstur LBP tiap gambar, diubah jadi histogram ternormalisasi sebagai vektor fitur. Dilakukan untuk train, val, dan test menggunakan fungsi yang sama, dan divisualisasikan.

In [ ]:
# Pengaturan Parameter LBP
radius = 3
n_points = 8 * radius
METHOD = 'uniform'

def extract_lbp_features(images):
    lbp_features = []
    lbp_images = []

    for img in tqdm(images, desc="Ekstraksi LBP"):
        #Menghitung matriks tekstur LBP dari gambar
        lbp = local_binary_pattern(img, n_points, radius, METHOD)
        lbp_images.append(lbp)

        #Membuat histogram dari matriks LBP sebagai vektor fitur numerik
        n_bins = int(lbp.max() + 1)
        hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))

        #Menormalisasi histogram agar nilainya proporsional
        hist = hist.astype("float")
        hist /= (hist.sum() + 1e-7)

        lbp_features.append(hist)

    return np.array(lbp_features), lbp_images

print("Mengekstrak fitur LBP dari Data Latih (Training)...")
X_train_lbp, lbp_imgs_train = extract_lbp_features(X_train)

print("\nMengekstrak fitur LBP dari Data Validasi (Validation)...")
X_val_lbp, _ = extract_lbp_features(X_val)

print("\nMengekstrak fitur LBP dari Data Uji (Testing)...")
X_test_lbp, _ = extract_lbp_features(X_test)

print(f"\nDimensi vektor fitur LBP (Training): {X_train_lbp.shape}")

#MENAMPILKAN OUTPUT FOTO (REAL & AI)
#Mencari indeks gambar pertama untuk kelas Real (0) dan AI (1) di data latih
idx_real = np.where(y_train == 0)[0][0]
idx_ai = np.where(y_train == 1)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#Foto Pola LBP - Wajah Asli (Real)
axes[0].imshow(lbp_imgs_train[idx_real], cmap='gray')
axes[0].set_title('Real - Tekstur LBP')
axes[0].axis('off')

#Foto Pola LBP - Wajah Buatan (AI)
axes[1].imshow(lbp_imgs_train[idx_ai], cmap='gray')
axes[1].set_title('AI - Tekstur LBP')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# **Ekstraksi Fitur HOG (Histogram of Oriented Gradients)**
Menghitung gradien arah tepi gambar (bentuk/kontur wajah).

In [ ]:
# Pengaturan Parameter HOG standar untuk citra wajah
ORIENTATIONS = 9
PIXELS_PER_CELL = (16, 16)
CELLS_PER_BLOCK = (2, 2)

def extract_hog_features(images):
    hog_features = []

    for img in tqdm(images, desc="Ekstraksi HOG"):
        # Menghitung Histogram of Oriented Gradients
        fd = hog(img,
                 orientations=ORIENTATIONS,
                 pixels_per_cell=PIXELS_PER_CELL,
                 cells_per_block=CELLS_PER_BLOCK,
                 block_norm='L2-Hys',
                 visualize=False) # visualize=False untuk mempercepat ekstraksi massal

        hog_features.append(fd)

    return np.array(hog_features)

print("Mengekstrak fitur HOG dari Data Latih (Training)...")
X_train_hog = extract_hog_features(X_train)

print("\nMengekstrak fitur HOG dari Data Validasi (Validation)...")
X_val_hog = extract_hog_features(X_val)

print("\nMengekstrak fitur HOG dari Data Uji (Testing)...")
X_test_hog = extract_hog_features(X_test)

print(f"\nDimensi vektor fitur HOG (Training): {X_train_hog.shape}")

#MENAMPILKAN OUTPUT FOTO (REAL & AI)
#Mengambil indeks gambar dari kelas Real (0) dan AI (1) di data latih
idx_real = np.where(y_train == 0)[0][0]
idx_ai = np.where(y_train == 1)[0][0]

# Ekstraksi khusus dengan visualize=True untuk mendapatkan gambar gradien
_, hog_img_real = hog(X_train[idx_real], orientations=ORIENTATIONS, pixels_per_cell=PIXELS_PER_CELL,
                      cells_per_block=CELLS_PER_BLOCK, block_norm='L2-Hys', visualize=True)

_, hog_img_ai = hog(X_train[idx_ai], orientations=ORIENTATIONS, pixels_per_cell=PIXELS_PER_CELL,
                    cells_per_block=CELLS_PER_BLOCK, block_norm='L2-Hys', visualize=True)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#Foto Kontur HOG - Real
axes[0].imshow(hog_img_real, cmap='gray')
axes[0].set_title('Real - Kontur HOG')
axes[0].axis('off')

#Foto Kontur HOG - AI
axes[1].imshow(hog_img_ai, cmap='gray')
axes[1].set_title('AI - Kontur HOG')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# **Ekstraksi Fitur Geometri (68 Landmark Wajah)**
Menggunakan detector + predictor Dlib untuk menemukan 68 titik koordinat wajah (mata, hidung, mulut, rahang, dst).

In [ ]:
#Inisialisasi detector dan predictor dari Dlib
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(model_path)

def extract_landmarks(images):
    landmarks_features = []
    landmarks_visuals = []

    for img in tqdm(images, desc="Ekstraksi Geometri"):
        img_uint8 = cv2.convertScaleAbs(img)
        faces = detector(img_uint8, 1)

        if len(faces) > 0:
            # Mengambil prediksi geometri pada wajah pertama yang terdeteksi
            shape = predictor(img_uint8, faces[0])
            coords = np.zeros((68, 2), dtype="int")
            for i in range(0, 68):
                coords[i] = (shape.part(i).x, shape.part(i).y)
            landmarks_features.append(coords.flatten())
            landmarks_visuals.append(coords)
        else:
            landmarks_features.append(np.zeros(136))
            landmarks_visuals.append(np.zeros((68, 2), dtype="int"))

    return np.array(landmarks_features), landmarks_visuals

print("Mengekstrak Landmark dari Data Latih (Training)...")
X_train_lm, lm_vis_train = extract_landmarks(X_train)

print("\nMengekstrak Landmark dari Data Validasi (Validation)...")
X_val_lm, _ = extract_landmarks(X_val)

print("\nMengekstrak Landmark dari Data Uji (Testing)...")
X_test_lm, _ = extract_landmarks(X_test)

print(f"\nDimensi vektor fitur Geometri (Training): {X_train_lm.shape}")

#MENAMPILKAN OUTPUT FOTO (REAL & AI)
idx_real = np.where(y_train == 0)[0][0]
idx_ai = np.where(y_train == 1)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# Foto Geometri - Wajah Asli (Real)
axes[0].imshow(X_train[idx_real], cmap='gray')
coords_real = lm_vis_train[idx_real]
if np.any(coords_real):
    axes[0].scatter(coords_real[:, 0], coords_real[:, 1], s=10, c='red', marker='o')
axes[0].set_title('Real - 68 Titik Koordinat')
axes[0].axis('off')

# Foto Geometri - Wajah Buatan (AI)
axes[1].imshow(X_train[idx_ai], cmap='gray')
coords_ai = lm_vis_train[idx_ai]
if np.any(coords_ai):
    axes[1].scatter(coords_ai[:, 0], coords_ai[:, 1], s=10, c='red', marker='o')
axes[1].set_title('AI - 68 Titik Koordinat')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# **Gabungkan Fitur + Standarisasi + PCA**
Tiga jenis fitur (LBP + HOG + Landmark) digabung jadi satu vektor besar per gambar. Distandarisasi (Z-score) supaya skalanya sebanding, lalu direduksi dengan PCA menjadi 100 komponen utama. Scaler dan PCA di-fit hanya dari data training.

In [ ]:
print("1. Menyatukan fitur LBP, HOG, dan Geometri...")
X_train_concat = np.hstack((X_train_lbp, X_train_hog, X_train_lm))
X_val_concat = np.hstack((X_val_lbp, X_val_hog, X_val_lm))
X_test_concat = np.hstack((X_test_lbp, X_test_hog, X_test_lm))

print(f"   Dimensi sebelum PCA: {X_train_concat.shape[1]} fitur")

print("\n2. Melakukan Standarisasi Data (Z-Score Scaling)...")
#Menyamakan skala angka antara LBP, HOG, dan Koordinat Geometri
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_concat)
X_val_scaled = scaler.transform(X_val_concat)
X_test_scaled = scaler.transform(X_test_concat)

print("\n3. Menerapkan Principal Component Analysis (PCA)...")
#Memangkas ribuan fitur menjadi hanya 100 komponen utama paling penting
pca = PCA(n_components=100, random_state=42)
X_train_final = pca.fit_transform(X_train_scaled)
X_val_final = pca.transform(X_val_scaled)
X_test_final = pca.transform(X_test_scaled)

#Menghitung seberapa besar varians informasi yang berhasil dipertahankan
explained_variance = np.sum(pca.explained_variance_ratio_) * 100

print("\n=== HASIL AKHIR TAHAP 9 (PENGGABUNGAN & PCA) ===")
print(f"Dimensi Data Latih Akhir    : {X_train_final.shape}")
print(f"Dimensi Data Validasi Akhir : {X_val_final.shape}")
print(f"Dimensi Data Uji Akhir      : {X_test_final.shape}")
print(f"Total Informasi Retained    : {explained_variance:.2f}% varians dipertahankan")

# **Training 4 Model**
Melatih SVM (kernel linear), KNN (k=5), AdaBoost, dan Gradient Boosting menggunakan fitur hasil PCA dari data training.

In [ ]:
#Menambahkan probability=True pada SVM
svm_model = SVC(kernel='linear', probability=True, random_state=42,)
knn_model = KNeighborsClassifier(n_neighbors=5)
ada_model = AdaBoostClassifier(n_estimators=50, random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=50, random_state=42)

models = {
    "Support Vector Machine (SVM)": svm_model,
    "K-Nearest Neighbors (KNN)": knn_model,
    "AdaBoost": ada_model,
    "Gradient Boosting": gb_model
}

trained_models = {}

print("Memulai proses pelatihan model pada data hasil PCA\n")

for name, model in models.items():
    print(f"--> Sedang melatih {name}...")
    start_time = time.time()

    #Melatih model menggunakan matriks 100 dimensi hasil PCA
    model.fit(X_train_final, y_train)

    end_time = time.time()
    duration = end_time - start_time

    trained_models[name] = model
    print(f"    [Selesai] Waktu pelatihan: {duration:.2f} detik\n")

print("SEMUA MODEL BERHASIL DILATIH")

# **Evaluasi Train vs Validasi**
Menghitung akurasi tiap model di data Training dan Validasi, lalu memberi label status ("Optimal/Stabil", "Indikasi Overfitting", dst) berdasarkan selisih akurasinya. Ini pengecekan overfitting versi awal.

In [ ]:
val_results = {}

print("Menguji model menggunakan 20% Data Validasi...\n")

for name, model in trained_models.items():
    #Mengukur performa di ruang kelas (Data Latih)
    y_train_pred = model.predict(X_train_final)
    acc_train = accuracy_score(y_train, y_train_pred)

    #Mengukur performa di ujian bayangan (Data Validasi)
    y_val_pred = model.predict(X_val_final)
    acc_val = accuracy_score(y_val, y_val_pred)

    #Menentukan status: Jika selisih Akurasi Training dan Validasi > 10%, kemungkinan Overfitting
    if acc_train - acc_val > 0.10:
        status = "Indikasi Overfitting"
    elif acc_val > acc_train:
        status = "Underfitting / Anomali Data"
    else:
        status = "Optimal (Stabil)"

    val_results[name] = {
        "Akurasi Training": f"{acc_train*100:.2f}%",
        "Akurasi Validasi": f"{acc_val*100:.2f}%",
        "Selisih (Gap)": f"{(acc_train - acc_val)*100:.2f}%",
        "Status Generalisasi": status
    }

# Menampilkan hasil komparasi
df_val = pd.DataFrame(val_results).T
print("HASIL UJI VALIDASI")
print(df_val)

# **K-Fold Cross Validation (k=5)**
Menggabungkan Train+Val jadi satu "development set" (Test tetap disisihkan), lalu melakukan 5-fold CV. Scaler dan PCA di-refit ulang di setiap fold. Menghasilkan rata-rata akurasi, presisi, recall, F1 per model.

In [ ]:
#K-FOLD CROSS VALIDATION (dengan Precision, Recall, F1-Score)

from sklearn.model_selection import StratifiedKFold

X_dev_concat = np.vstack((X_train_concat, X_val_concat))
y_dev = np.hstack((y_train, y_val))

print(f"Total data untuk K-Fold CV: {X_dev_concat.shape[0]} sampel\n")

k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

models_for_cv = {
    "Support Vector Machine (SVM)": lambda: SVC(kernel='linear', probability=True, random_state=42),
    "K-Nearest Neighbors (KNN)": lambda: KNeighborsClassifier(n_neighbors=5),
    "AdaBoost": lambda: AdaBoostClassifier(n_estimators=50, random_state=42),
    "Gradient Boosting": lambda: GradientBoostingClassifier(n_estimators=50, random_state=42)
}

kfold_results = {}

for name, model_fn in models_for_cv.items():
    fold_acc, fold_prec, fold_rec, fold_f1 = [], [], [], []
    print(f"--- {name} ---")

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_dev_concat, y_dev), start=1):
        X_fold_train_raw = X_dev_concat[train_idx]
        X_fold_val_raw = X_dev_concat[val_idx]
        y_fold_train = y_dev[train_idx]
        y_fold_val = y_dev[val_idx]

        fold_scaler = StandardScaler()
        X_fold_train_scaled = fold_scaler.fit_transform(X_fold_train_raw)
        X_fold_val_scaled = fold_scaler.transform(X_fold_val_raw)

        fold_pca = PCA(n_components=100, random_state=42)
        X_fold_train_pca = fold_pca.fit_transform(X_fold_train_scaled)
        X_fold_val_pca = fold_pca.transform(X_fold_val_scaled)

        model = model_fn()
        model.fit(X_fold_train_pca, y_fold_train)
        y_fold_pred = model.predict(X_fold_val_pca)

        acc = accuracy_score(y_fold_val, y_fold_pred)
        prec = precision_score(y_fold_val, y_fold_pred)
        rec = recall_score(y_fold_val, y_fold_pred)
        f1 = f1_score(y_fold_val, y_fold_pred)

        fold_acc.append(acc)
        fold_prec.append(prec)
        fold_rec.append(rec)
        fold_f1.append(f1)

        print(f"  Fold {fold_idx}: Acc={acc*100:.2f}%, Presisi={prec*100:.2f}%, Recall={rec*100:.2f}%, F1={f1*100:.2f}%")

    kfold_results[name] = {
        "Mean Accuracy": f"{np.mean(fold_acc)*100:.2f}% (±{np.std(fold_acc)*100:.2f}%)",
        "Mean Presisi": f"{np.mean(fold_prec)*100:.2f}% (±{np.std(fold_prec)*100:.2f}%)",
        "Mean Recall": f"{np.mean(fold_rec)*100:.2f}% (±{np.std(fold_rec)*100:.2f}%)",
        "Mean F1-Score": f"{np.mean(fold_f1)*100:.2f}% (±{np.std(fold_f1)*100:.2f}%)",
    }
    print(f"  => Rata-rata Acc: {np.mean(fold_acc)*100:.2f}% (+/- {np.std(fold_acc)*100:.2f}%)\n")

df_kfold = pd.DataFrame(kfold_results).T
print("=== RINGKASAN HASIL K-FOLD CROSS VALIDATION (k=5) ===")
print(df_kfold)

# **Evaluasi Data Uji (Test Set)**
Menghitung akurasi Training, Validasi, dan Uji (Test) sekaligus, plus precision/recall/F1 khusus untuk test set / data uji.

In [ ]:
test_results = {}

print("Menguji model menggunakan 10% Data Uji (Testing)...\n")

for name, model in trained_models.items():
    y_test_pred = model.predict(X_test_final)

    acc_test = accuracy_score(y_test, y_test_pred)
    prec_test = precision_score(y_test, y_test_pred)
    rec_test = recall_score(y_test, y_test_pred)
    f1_test = f1_score(y_test, y_test_pred)

    # Ambil ulang akurasi training & validasi untuk melihat gap 3 tahap sekaligus
    y_train_pred = model.predict(X_train_final)
    acc_train = accuracy_score(y_train, y_train_pred)

    y_val_pred = model.predict(X_val_final)
    acc_val = accuracy_score(y_val, y_val_pred)

    test_results[name] = {
        "Akurasi Training": f"{acc_train*100:.2f}%",
        "Akurasi Validasi": f"{acc_val*100:.2f}%",
        "Akurasi Uji (Test)": f"{acc_test*100:.2f}%",
        "Presisi Uji": f"{prec_test*100:.2f}%",
        "Recall Uji": f"{rec_test*100:.2f}%",
        "F1-Score Uji": f"{f1_test*100:.2f}%"
    }

df_test = pd.DataFrame(test_results).T
print("=== HASIL UJI PADA DATA TESTING (10%, Held-Out) ===")
print(df_test)


# **Confusion Matrix & ROC Curve**
Visualisasi performa keempat model terhadap X_test_final: confusion matrix (atas) dan kurva ROC + AUC (bawah) untuk tiap model.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, (name, model) in enumerate(trained_models.items()):
    # CONFUSION MATRIX ---
    y_pred = model.predict(X_test_final)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, i],
                xticklabels=['Real (0)', 'AI (1)'], yticklabels=['Real (0)', 'AI (1)'],
                annot_kws={"size": 12})
    axes[0, i].set_title(f'Confusion Matrix\n{name}')
    axes[0, i].set_ylabel('Label Asli')
    axes[0, i].set_xlabel('Prediksi Model')

    # Mengambil nilai probabilitas prediksi kelas positif (AI)
    y_score = model.predict_proba(X_test_final)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)

    axes[1, i].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
    axes[1, i].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[1, i].set_xlim([0.0, 1.0])
    axes[1, i].set_ylim([0.0, 1.05])
    axes[1, i].set_xlabel('False Positive Rate')
    axes[1, i].set_ylabel('True Positive Rate')
    axes[1, i].set_title(f'Kurva ROC\n{name}')
    axes[1, i].legend(loc="lower right")

plt.tight_layout()
plt.show()

# **Load Dataset Crosscheck**
Memuat dataset terpisah (dataset_crosscheck_final) — 100 foto Real + 100 foto AI dari sumber yang berbeda dari dataset training, untuk menguji generalisasi model ke data benar-benar baru/independen.

In [ ]:
crosscheck_base = '/content/drive/MyDrive/Tubes_RF/dataset_crosscheck_final'
crosscheck_real_path = os.path.join(crosscheck_base, 'real')
crosscheck_ai_path = os.path.join(crosscheck_base, 'ai')

print("Memuat dataset Crosscheck Real...")
cc_real_images, cc_real_labels = load_images_only(crosscheck_real_path, label=0)

print("\nMemuat dataset Crosscheck AI...")
cc_ai_images, cc_ai_labels = load_images_only(crosscheck_ai_path, label=1)

X_crosscheck = np.array(cc_real_images + cc_ai_images)
y_crosscheck = np.array(cc_real_labels + cc_ai_labels)
print(f"\nTotal data crosscheck: {len(X_crosscheck)} gambar")

# **Ekstraksi Fitur untuk Crosscheck**
Menerapkan fungsi ekstraksi LBP, HOG, dan Landmark yang sama persis ke data crosscheck.

In [ ]:
#Ekstraksi fitur - pakai fungsi yang SAMA PERSIS (extract_lbp_features, extract_hog_features, extract_landmarks)
print("Mengekstrak fitur LBP dari Data Crosscheck...")
X_crosscheck_lbp, _ = extract_lbp_features(X_crosscheck)

print("\nMengekstrak fitur HOG dari Data Crosscheck...")
X_crosscheck_hog = extract_hog_features(X_crosscheck)

print("\nMengekstrak fitur Landmark dari Data Crosscheck...")
X_crosscheck_lm, _ = extract_landmarks(X_crosscheck)

print(f"\nDimensi fitur - LBP: {X_crosscheck_lbp.shape}, HOG: {X_crosscheck_hog.shape}, Landmark: {X_crosscheck_lm.shape}")

# **Transform Fitur Crosscheck**
Gabungkan fitur crosscheck, lalu transform pakai scaler dan pca yang sudah di-fit dari data training, supaya representasi fiturnya konsisten dengan yang dipelajari model.

In [ ]:
#Gabungkan fitur + transform pakai scaler & PCA yang SUDAH DI-FIT dari data training

X_crosscheck_concat = np.hstack((X_crosscheck_lbp, X_crosscheck_hog, X_crosscheck_lm))
X_crosscheck_scaled = scaler.transform(X_crosscheck_concat)
X_crosscheck_final = pca.transform(X_crosscheck_scaled)

print(f"Dimensi akhir data crosscheck setelah PCA: {X_crosscheck_final.shape}")

# **Prediksi & Evaluasi Crosscheck**
Menjalankan keempat model ke data crosscheck, menghitung akurasi/presisi/recall/F1. Di sinilah terlihat akurasi anjlok ke ~50-57% untuk semua model, bukti adanya generalization gap

In [ ]:
#Prediksi dengan keempat model yang sudah dilatih, dan evaluasi hasilnya

crosscheck_results = {}

print("=== HASIL PREDIKSI PADA DATA CROSSCHECK (DI LUAR DATASET TRAINING) ===\n")

for name, model in trained_models.items():
    y_cc_pred = model.predict(X_crosscheck_final)
    acc_cc = accuracy_score(y_crosscheck, y_cc_pred)
    prec_cc = precision_score(y_crosscheck, y_cc_pred)
    rec_cc = recall_score(y_crosscheck, y_cc_pred)
    f1_cc = f1_score(y_crosscheck, y_cc_pred)

    crosscheck_results[name] = {
        "Akurasi": f"{acc_cc*100:.2f}%",
        "Presisi": f"{prec_cc*100:.2f}%",
        "Recall": f"{rec_cc*100:.2f}%",
        "F1-Score": f"{f1_cc*100:.2f}%"
    }


df_crosscheck = pd.DataFrame(crosscheck_results).T
print("=== RINGKASAN HASIL CROSSCHECK ===")
print(df_crosscheck)

# **Confusion Matrix Crosscheck**
Visualisasi confusion matrix keempat model khusus untuk data crosscheck.

In [ ]:
#Confusion matrix visual untuk crosscheck
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, (name, model) in enumerate(trained_models.items()):
    y_cc_pred = model.predict(X_crosscheck_final)
    cm = confusion_matrix(y_crosscheck, y_cc_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[i],
                xticklabels=['Real (0)', 'AI (1)'], yticklabels=['Real (0)', 'AI (1)'])
    axes[i].set_title(f'Crosscheck\n{name}')
    axes[i].set_ylabel('Label Asli')
    axes[i].set_xlabel('Prediksi Model')

plt.tight_layout()
plt.show()